# Windowing and Feature Extraction

We segment sEMG into overlapping windows and extract a compact set of time-domain features.
Each subject is processed independently so we can train a single model per subject.


## What you will do
- Build within-subject train/test splits based on repetitions.
- Generate 200 ms sliding windows with a 10 ms step.
- Subsample the training set to reduce redundant windows.
- Extract MAV, WL, and VAR features per channel.
- Save a per-subject feature dataset for modeling.


## Configuration

In [1]:
import sys
from pathlib import Path

import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler

# Resolve the project root (one level above this notebook/script).
# This enables absolute imports like "from src.datasets import ..."
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.datasets import build_subject_dataset

In [ ]:
# In NinaPro DB1, the sEMG stream is sampled at 100 Hz.
FS_HZ = 100

# Sliding-window classification:
WIN_MS = 200  # window length in milliseconds
STEP_MS = 10  # hop / increment in milliseconds (paper uses 10 ms)


# Convert ms → samples
WIN_SAMPLES = int(FS_HZ * WIN_MS / 1000)
HOP_SAMPLES = int(FS_HZ * STEP_MS / 1000)

# The train vs test set are split based on repetitions.
TRAIN_REPS = {1, 3, 4, 5, 9}
TEST_REPS = {2, 6, 7, 8, 10}

DB_PATH = Path("../data/processed/db1_a1.pkl")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Load prebuilt DB dictionary
This uses the DB1 A1 pickle built in the notebook `01_data_loading_and_overview`.


In [3]:
with DB_PATH.open("rb") as f:
    db = pickle.load(f)

subject_ids = sorted(db.keys())
print(subject_ids)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]


## Build a within-subject dataset
Each subject gets its own train/test split based on repetitions.

### Methodology note: movement ID remapping
NinaPro DB1 labels restart at 1 in each exercise (with rest = 0).
Because we concatenate exercises into a single dataset, we remap movement IDs to a global, contiguous range:
- Rest stays **0**.
- Exercise 1 movements map to **1–12**.
- Exercise 2 movements map to **13–29**.
- Exercise 3 movements map to **30–52**.

This keeps label semantics consistent across exercises and makes results reproducible.

In [4]:
SUBJECT_ID = 1
dataset_raw = build_subject_dataset(
    SUBJECT_ID,
    db,
    win_samples=WIN_SAMPLES,
    hop_samples=HOP_SAMPLES,
    train_reps=TRAIN_REPS,
    test_reps=TEST_REPS,
)
dataset_raw["X_train"].shape, dataset_raw["X_test"].shape

((253857, 30), (216565, 30))

## Subsampling of the training set
The training set is regularly subsampled by a factor of 10 by keeping every 10-th training sample.
This step reduces the total number of training samples and speeds up experimentation.
Because the data is generated using a highly overlapping sliding window (one window every 10 ms), neighboring samples are very similar. Regular subsampling removes this redundancy while keeping the overall data distribution unchanged.
The test set is not subsampled, so evaluation is still performed on the full data.


In [5]:
def subsample_train(
    dataset: dict[str, np.ndarray],
    factor: int = 10,
) -> dict[str, np.ndarray]:
    """
    Subsample the TRAIN split by taking every `factor`-th sample.
    Test set is left untouched.
    """

    idx = np.arange(0, len(dataset["y_train"]), factor)

    return {
        **dataset,
        "X_train": dataset["X_train"][idx],
        "y_train": dataset["y_train"][idx],
    }


dataset_sub = subsample_train(dataset_raw, factor=10)

print("Before subsampling:")
print(f"  Train samples: {len(dataset_raw['y_train']):>8,}")
print(f"  Test samples : {len(dataset_raw['y_test']):>8,}")

print("After subsampling:")
print(f"  Train samples: {len(dataset_sub['y_train']):>8,}")
print(f"  Test samples : {len(dataset_sub['y_test']):>8,}")


Before subsampling:
  Train samples:  253,857
  Test samples :  216,565
After subsampling:
  Train samples:   25,386
  Test samples :  216,565


In [ ]:
import numpy as np


def print_label_counts(y, title):
    unique = np.unique(y)
    total = len(y)
    print("\n" + "=" * 60)
    print(f"Sample Counts per Label ({title})")
    print("=" * 60)
    for label in sorted(unique):
        count = int(np.sum(y == label))
        label_name = "REST" if label == 0 else f"Movement {label}"
        print(
            f"Label {label:2d} ({label_name:15s}): {count:6d} samples ({count / total * 100:5.1f}%)"
        )


print_label_counts(dataset_sub["y_train"], "Train")
print_label_counts(dataset_sub["y_test"], "Test")



Sample Counts per Label (Train)
Label  0 (REST           ):  16315 samples ( 64.3%)
Label  1 (Movement 1     ):    189 samples (  0.7%)
Label  2 (Movement 2     ):    113 samples (  0.4%)
Label  3 (Movement 3     ):    231 samples (  0.9%)
Label  4 (Movement 4     ):    153 samples (  0.6%)
Label  5 (Movement 5     ):    195 samples (  0.8%)
Label  6 (Movement 6     ):    155 samples (  0.6%)
Label  7 (Movement 7     ):    132 samples (  0.5%)
Label  8 (Movement 8     ):    148 samples (  0.6%)
Label  9 (Movement 9     ):    122 samples (  0.5%)
Label 10 (Movement 10    ):    180 samples (  0.7%)
Label 11 (Movement 11    ):    117 samples (  0.5%)
Label 12 (Movement 12    ):    153 samples (  0.6%)
Label 13 (Movement 13    ):    154 samples (  0.6%)
Label 14 (Movement 14    ):    140 samples (  0.6%)
Label 15 (Movement 15    ):    164 samples (  0.6%)
Label 16 (Movement 16    ):    141 samples (  0.6%)
Label 17 (Movement 17    ):    131 samples (  0.5%)
Label 18 (Movement 18    ):    

## Feature scaling (within-subject)
We standardize using training data only, then apply the same transform to test data.


In [7]:
scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(dataset_sub["X_train"])

X_train = scaler.transform(dataset_sub["X_train"]).astype(np.float32)
X_test = scaler.transform(dataset_sub["X_test"]).astype(np.float32)

dataset = {
    "X_train": X_train,
    "y_train": dataset_sub["y_train"],
    "X_test": X_test,
    "y_test": dataset_sub["y_test"],
    "scaler": {
        "mean": scaler.mean_.astype(np.float32),
        "scale": scaler.scale_.astype(np.float32),
    },
    "meta": {
        "subject_id": SUBJECT_ID,
        "features": ["mav", "wl", "var"],
        "win_samples": WIN_SAMPLES,
        "hop_samples": HOP_SAMPLES,
        "train_reps": sorted(list(TRAIN_REPS)),
        "test_reps": sorted(list(TEST_REPS)),
        "fs_hz": FS_HZ,
    },
}


## Save per-subject dataset
We store one dataset per subject to train a single model per subject.


In [ ]:
out_path = (
    OUT_DIR / f"db1_subject{SUBJECT_ID:02d}_win{WIN_SAMPLES}_hop{HOP_SAMPLES}.pkl"
)
with out_path.open("wb") as f:
    pickle.dump(dataset, f)

out_path

PosixPath('../data/processed/db1_subject01_win20_hop1.pkl')

## Next step
Use the per-subject dataset to baseline models.
